[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vpasumarthi/aimd-tutorial/blob/main/notebooks/03_constrained_md.ipynb)

# Constrained MD and Blue-Moon Ensemble

This tutorial covers constrained molecular dynamics and the Blue-Moon ensemble method for computing free energy profiles along a reaction coordinate. This is the standard AIMD approach for mapping free energy barriers of chemical reactions.

**Learning objectives:**
- Understand the theory behind constrained MD and thermodynamic integration
- Set up Blue-Moon ensemble calculations in VASP (ICONST, LBLUEOUT)
- Parse constraint forces from REPORT files
- Integrate the mean force to obtain a free energy profile
- Assess convergence of the mean constraint force

In [ ]:
# --- Run this cell first in Google Colab ---
import sys
if 'google.colab' in sys.modules:
    %pip install -q ase pymatgen nglview scipy
    from google.colab import output
    output.enable_custom_widget_manager()

## Background and Theory

Rare events (chemical reactions, phase transitions, conformational changes) happen on timescales far longer than accessible by direct AIMD. Constrained MD overcomes this by fixing a chosen reaction coordinate $\xi$ at a series of values and sampling the orthogonal degrees of freedom. The **Blue-Moon ensemble** provides the rigorous statistical-mechanical framework for extracting the free energy gradient from constrained simulations.

### Thermodynamic Integration

The free energy along the reaction coordinate $\xi$ is obtained by integrating the mean constraint force:

$$\Delta F(\xi) = \int_{\xi_0}^{\xi} \left\langle f(\xi') \right\rangle_{\xi'} \, d\xi'$$

where the mean force at each constrained value is:

$$\langle f(\xi) \rangle = \frac{\langle Z^{-1/2} (\lambda + G k_B T) \rangle_\xi}{\langle Z^{-1/2} \rangle_\xi}$$

Here:
- $\lambda$ is the Lagrange multiplier (the force needed to maintain the constraint)
- $Z = \nabla \xi \cdot M^{-1} \cdot \nabla \xi$ is the mass-metric tensor
- $G$ is a geometric correction term

For a simple bond distance constraint between two atoms $i$ and $j$, $Z$ and $G$ simplify considerably: $|Z|^{1/2} = (1/m_i + 1/m_j)^{1/2}$ (a constant) and $G = 0$. The mean force then reduces to $\langle f \rangle = \langle \lambda \rangle$.

### Practical Workflow

1. Choose the reaction coordinate $\xi$ and define constraint windows
2. For each window: run constrained AIMD at fixed $\xi$ and collect the constraint force
3. Compute the mean force at each window
4. Integrate to get $\Delta F(\xi)$

## Example System

We demonstrate the method using a proton transfer reaction in liquid water:

$$\text{H}_2\text{O} + \text{H}_2\text{O} \to \text{OH}^- + \text{H}_3\text{O}^+$$

The reaction coordinate is the O-H distance $d(\text{O}_\text{donor}\text{-H}^*)$ of the transferring proton. We constrain this distance at values from 1.0 to 2.0 Angstrom in 11 windows. At each window, the system runs NVT AIMD with the constraint active, and VASP writes the Lagrange multiplier to the REPORT file.

## VASP Input Files

### INCAR

```
SYSTEM = Constrained MD - Blue Moon proton transfer

# Electronic structure
PREC    = Normal
ENCUT   = 400
ALGO    = VeryFast
EDIFF   = 1E-5
ISMEAR  = 0
SIGMA   = 0.1
LREAL   = Auto
NELMIN  = 4

# Molecular dynamics
IBRION  = 0
MDALGO  = 2          # Nose-Hoover thermostat
SMASS   = 1.0
POTIM   = 1.0        # 1 fs timestep
NSW     = 3000       # 3 ps per window
TEBEG   = 400
TEEND   = 400
ISIF    = 2

# Blue Moon output
LBLUEOUT = .TRUE.    # Write constraint data to REPORT file

# Output
NWRITE  = 0
NBLOCK  = 1
LWAVE   = .FALSE.
LCHARG  = .FALSE.
```

**Key setting:** `LBLUEOUT = .TRUE.` instructs VASP to write the Lagrange multiplier, the mass-metric tensor determinant $|Z|$, the geometric correction $G$, and the corrected mean force to the REPORT file at each MD step.

### ICONST File

The ICONST file defines the geometric constraints. Each line specifies:
```
TYPE  ATOM1  ATOM2  [ATOM3  [ATOM4]]  FLAG
```

- `TYPE`: `R` = bond distance, `A` = angle, `T` = torsion
- `FLAG`: `0` = unconstrained/unmonitored, `5` = monitored only, `7` = constrained
- Atom indices are **1-based** (matching POSCAR ordering)

For our O-H distance constraint (atoms 1 and 3 in the POSCAR):
```
R 1 3 7
```

This constrains the distance between atom 1 (O$_\text{donor}$) and atom 3 (H$^*$) at whatever value is present in the initial POSCAR for that window.

**Workflow for multiple windows:** Create a separate directory for each constraint value. In each directory, place the same INCAR, KPOINTS, and POTCAR, but use a POSCAR where the O-H distance has been adjusted to the target value. VASP reads the initial geometry and constrains at that distance.

```bash
# Example directory structure
constrained_md/
  window_1.00/   # d(O-H) = 1.00 A
    INCAR  POSCAR  KPOINTS  POTCAR  ICONST
  window_1.10/   # d(O-H) = 1.10 A
    INCAR  POSCAR  KPOINTS  POTCAR  ICONST
  ...
  window_2.00/   # d(O-H) = 2.00 A
    INCAR  POSCAR  KPOINTS  POTCAR  ICONST
```

---
## Analysis

After all windows complete, copy the REPORT file from each window into `data/constrained_md/`. Name them by their constraint value, e.g. `REPORT_1.00`, `REPORT_1.10`, etc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid

plt.rcParams.update({
    'figure.figsize': (6, 4),
    'font.size': 12,
    'axes.linewidth': 1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.width': 1.2,
    'ytick.major.width': 1.2,
})

In [ ]:
def parse_report_file(filepath):
    """Parse Blue Moon output from a VASP REPORT file.

    Extracts the corrected mean force contribution at each MD step:
        |Z|^(-1/2) * (lambda + G*kT)
    and the normalization factor |Z|^(-1/2).

    Returns
    -------
    lambda_vals : ndarray
        Raw Lagrange multipliers (eV/Angstrom).
    z_inv_sqrt : ndarray
        |Z|^(-1/2) values.
    corrected_force : ndarray
        |Z|^(-1/2) * (lambda + G*kT) values (eV/Angstrom).
    """
    lambda_vals = []
    z_inv_sqrt = []
    corrected_force = []

    with open(filepath) as f:
        for line in f:
            # Blue Moon lines contain "cc>" prefix
            if 'cc>' in line and 'lambda' not in line.lower():
                parts = line.split()
                # Format: cc> lambda  |z|^(-1/2)  GkT  |z|^(-1/2)*(lambda+GkT)
                try:
                    idx = parts.index('cc>') + 1
                    lam = float(parts[idx])
                    z_inv = float(parts[idx + 1])
                    force_corr = float(parts[idx + 3])
                    lambda_vals.append(lam)
                    z_inv_sqrt.append(z_inv)
                    corrected_force.append(force_corr)
                except (ValueError, IndexError):
                    continue

    return np.array(lambda_vals), np.array(z_inv_sqrt), np.array(corrected_force)

In [ ]:
# Define the constraint windows
xi_values = np.arange(1.0, 2.05, 0.1)  # O-H distances in Angstrom

# Load REPORT files for each window
equil_steps = 500  # steps to discard per window

mean_forces = []
force_errors = []

for xi in xi_values:
    filepath = f'data/constrained_md/REPORT_{xi:.2f}'
    lam, z_inv, f_corr = parse_report_file(filepath)

    # Discard equilibration
    lam = lam[equil_steps:]
    z_inv = z_inv[equil_steps:]
    f_corr = f_corr[equil_steps:]

    # Blue Moon mean force: <f_corr> / <z_inv>
    mean_f = np.mean(f_corr) / np.mean(z_inv)
    mean_forces.append(mean_f)

    # Standard error via block averaging (see convergence section below)
    n = len(f_corr)
    se = np.std(f_corr / np.mean(z_inv)) / np.sqrt(n)
    force_errors.append(se)

    print(f'xi = {xi:.2f} A: <f> = {mean_f:+.4f} eV/A '
          f'({n} samples after equil)')

mean_forces = np.array(mean_forces)
force_errors = np.array(force_errors)

### Mean Force Profile

Before integrating, inspect the mean constraint force as a function of $\xi$. The force should cross zero at stable states and transition states.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(xi_values, mean_forces, yerr=force_errors,
            fmt='o-', capsize=3, color='C0', markersize=5)
ax.axhline(0, ls='--', color='gray', lw=0.8)
ax.set_xlabel(r'$\xi$ ($\mathrm{\AA}$)')
ax.set_ylabel(r'Mean force (eV/$\mathrm{\AA}$)')
ax.set_title('Mean constraint force')
plt.tight_layout()
plt.show()

### Free Energy Profile

Integrate the mean force using the trapezoidal rule to get the free energy $\Delta F(\xi)$, referenced to the first window ($\xi_0$).

In [ ]:
# Integrate: Delta F(xi) = integral from xi_0 to xi of <f> d(xi')
delta_F = np.zeros(len(xi_values))
delta_F[1:] = cumulative_trapezoid(mean_forces, xi_values)

# Propagate errors through integration (simplified: assume independent windows)
dxi = np.diff(xi_values)
delta_F_err = np.zeros(len(xi_values))
for i in range(1, len(xi_values)):
    # Trapezoidal error propagation
    delta_F_err[i] = np.sqrt(
        delta_F_err[i-1]**2 +
        (dxi[i-1] / 2)**2 * (force_errors[i-1]**2 + force_errors[i]**2)
    )

fig, ax = plt.subplots(figsize=(6, 4))
ax.fill_between(xi_values, delta_F - delta_F_err, delta_F + delta_F_err,
                alpha=0.3, color='C0')
ax.plot(xi_values, delta_F, 'o-', color='C0', markersize=5)
ax.set_xlabel(r'$\xi$ ($\mathrm{\AA}$)')
ax.set_ylabel(r'$\Delta F$ (eV)')
ax.set_title('Free energy profile')
plt.tight_layout()
plt.show()

# Report barrier height
barrier = np.max(delta_F) - delta_F[0]
ts_xi = xi_values[np.argmax(delta_F)]
print(f'Free energy barrier: {barrier:.3f} eV ({barrier * 23.06:.1f} kcal/mol)')
print(f'Transition state at xi = {ts_xi:.2f} A')

### Convergence Analysis

The reliability of the free energy profile depends on whether the mean force has converged at each window. We check convergence by computing a running average and by block averaging to estimate the true statistical error (accounting for autocorrelation).

In [ ]:
def block_average_error(data, n_blocks=10):
    """Estimate the standard error of the mean using block averaging.

    Block averaging accounts for autocorrelation in the time series,
    giving a more reliable error estimate than naive standard error.
    """
    n = len(data)
    block_size = n // n_blocks
    if block_size < 1:
        return np.std(data) / np.sqrt(n)

    block_means = []
    for i in range(n_blocks):
        start = i * block_size
        end = start + block_size
        block_means.append(np.mean(data[start:end]))

    block_means = np.array(block_means)
    return np.std(block_means) / np.sqrt(n_blocks)


# Show convergence for a selected window
xi_check = 1.50  # Angstrom -- near the expected transition state
filepath = f'data/constrained_md/REPORT_{xi_check:.2f}'
lam, z_inv, f_corr = parse_report_file(filepath)
lam = lam[equil_steps:]

# Running average of the Lagrange multiplier
running_avg = np.cumsum(lam) / np.arange(1, len(lam) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Running average
steps = np.arange(len(lam))
ax1.plot(steps, lam, lw=0.3, alpha=0.5, color='C0', label='Instantaneous')
ax1.plot(steps, running_avg, lw=1.5, color='C3', label='Running average')
ax1.set_xlabel('MD step (after equilibration)')
ax1.set_ylabel(r'$\lambda$ (eV/$\mathrm{\AA}$)')
ax1.set_title(f'Constraint force at $\\xi$ = {xi_check} A')
ax1.legend(fontsize=9)

# Block averaging error vs block size
block_counts = np.arange(3, 30)
errors = [block_average_error(lam, nb) for nb in block_counts]
ax2.plot(len(lam) // block_counts, errors, 'o-', markersize=4, color='C0')
ax2.set_xlabel('Block size (steps)')
ax2.set_ylabel('Standard error of mean')
ax2.set_title('Block averaging convergence')

plt.tight_layout()
plt.show()

# Report block-averaged error
se_block = block_average_error(lam, n_blocks=10)
print(f'Block-averaged SE (10 blocks): {se_block:.4f} eV/A')
print(f'Naive SE: {np.std(lam)/np.sqrt(len(lam)):.4f} eV/A')

**Interpreting convergence:**
- The running average should plateau. If it is still drifting at the end, the window needs longer sampling.
- The block-averaged standard error should be larger than the naive SE (by a factor of $\sqrt{2\tau_{\text{corr}}/\Delta t}$ where $\tau_{\text{corr}}$ is the autocorrelation time). If the block error is much larger, increase the simulation length.
- A rule of thumb: aim for at least 10 autocorrelation times of sampling per window.

## Summary

In this tutorial you learned to:
1. Set up constrained AIMD in VASP using the ICONST file and `LBLUEOUT`
2. Organize and run multiple Blue-Moon ensemble windows
3. Parse the constraint force from VASP REPORT files
4. Compute the free energy profile by thermodynamic integration
5. Assess convergence using running averages and block averaging

**Next steps:**
- Use more complex reaction coordinates (bond angle, coordination number) for multi-step reactions
- Compare with metadynamics (`MDALGO = 11` in VASP) for high-dimensional free energy surfaces
- Add more windows in regions where the force changes rapidly for better integration accuracy
- Try the slow-growth approach (continuously varying constraint) as a quick initial scan before committing to Blue-Moon windows